1. Imports and configuration 

In [1]:

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [2]:
filename = os.path.join(os.getcwd(), "..", "data", "public2025 3.csv")
df = pd.read_csv(filename, header=0)
df.head()
df.shape


/var/folders/6x/1g6b_h6x3rggmc8pyzwyjq3w0000gn/T/ipykernel_67052/3312385562.py:2: DtypeWarning: Columns (0: GH14, 1: ppfs1460, 2: ppfs1461, 3: ppfs1462, 4: ppfs1463, 5: ppfs1613, 6: ppfs1464, 7: ppfs0549, 8: ppfs1493, 9: ppfs2342, 10: ppfs2309, 11: ppfs2340, 12: ppfs2343, 13: ppfs2344, 14: ppfs2345) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename, header=0)


(12934, 815)

In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 12934 entries, 0 to 12933
Columns: 815 entries, shedid to year
dtypes: float64(19), int64(370), object(1), str(425)
memory usage: 80.4+ MB


,shedid,duration,weight,weight_pop,panel_weight,panel_weight_pop,CG2,R3,M4,BK49A,...,E11_g_iflag,E12_a_iflag,E12_b_iflag,E12_c_iflag,E12_d_iflag,E12_e_iflag,E12_f_iflag,E12_g_iflag,CH2A_iflag,year
count,1.293400e+04,12934.000000,12934.000000,12934.000000,4419.000000,4419.000000,529.000000,3426.000000,5176.000000,6.720000e+02,...,12934.000000,12934.000000,12934.000000,12934.000000,12934.000000,12934.000000,12934.000000,12934.000000,12934.000000,12934.0
mean,2.023368e+08,12983.185712,0.999999,20451.455468,1.000000,59859.498757,310.395085,1345.832166,1806.100850,1.685096e+04,...,0.056131,0.003557,0.003247,0.002783,0.002706,0.002010,0.000387,0.006572,0.002861,2025.0
std,2.496523e+05,60579.334717,0.379342,7758.099154,0.371248,22222.775582,297.836601,949.061696,1395.170523,2.166922e+05,...,0.230184,0.059533,0.056894,0.052686,0.051951,0.044792,0.019659,0.080803,0.053411,0.0
min,2.013001e+08,423.000000,0.331900,6788.330300,0.181500,10866.651300,5.000000,0.000000,0.000000,1.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2025.0
25%,2.023001e+08,953.000000,0.750300,15344.733300,0.748800,44822.040700,105.000000,700.000000,1000.000000,1.552500e+02,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2025.0
50%,2.025004e+08,1275.000000,0.924500,18906.865400,0.887500,53126.342800,242.000000,1200.000000,1500.000000,5.000000e+02,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2025.0
75%,2.025036e+08,1963.000000,1.166800,23862.983700,1.186900,71049.029800,400.000000,1750.000000,2200.000000,1.800000e+03,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2025.0
max,2.025069e+08,898185.000000,3.488200,71339.356900,3.372100,201851.327000,2500.000000,9999.000000,19999.000000,5.000000e+06,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2025.0


In [4]:

## Identify all "iflag" columns
iflag_column = []
for c in df.columns:
    if c.endswith("_iflag"):
        iflag_column.append(c)
print(len(iflag_column))

357


Data validation 

In [6]:
for col in ["weight", "weight_pop", "shedid","pay_casheqv"]:
    print(col, col in df.columns)
print("\nTarget response counts:")
print(df["pay_casheqv"].value_counts(dropna=False))



weight True
weight_pop True
shedid True
pay_casheqv True

Target response counts:
pay_casheqv
Yes    8398
No     4536
Name: count, dtype: int64


Data replication 
-federal report found 63% adult cover $400 the emergency found
-0.5 weighted estimate is considered sucess 
-weight_pop as us adult population 


In [ ]:
# Total population weight
fedral_estimate=63.0
accepted_gap=0.5
total_weight = df["weight_pop"].sum()

# Weight for people who said Yes
yes_weight = df.loc[df["pay_casheqv"] == "Yes", "weight_pop"].sum() //who can cover the expense 

# Calculate weighted percentage
weighted_percent = (yes_weight / total_weight) * 100
our_gap=abs (weighted_percent-fedral_estimate)

print(f"fedral estimate is : {fedral_estimate:.2f}")
print(f"Our weighted  is : {weighted_percent:.2f}")
print(f"the gap is  : {our_gap:.2f}")



fedral estimate is : 63.00
Our weighted  is : 63.14
the gap is  : 0.14


In [10]:
print(df["pay_casheqv"].isnull().sum())

0


In [11]:
fragility = df["pay_casheqv"].map({
    "No": 1,
    "Yes": 0
})

if fragility.isna().any():
    raise ValueError(
        "The target contains a missing or unexpected response."
    )

population_weights = df["weight_pop"].copy()

print("Financial-fragility target counts:")
print(fragility.value_counts().sort_index())

print("\nNumber of population weights:", len(population_weights))
print("Number of missing population weights:", population_weights.isna().sum())

Financial-fragility target counts:
pay_casheqv
0    8398
1    4536
Name: count, dtype: int64

Number of population weights: 12934
Number of missing population weights: 0


In [12]:
NOT_ASKED = "__NOT_ASKED__"
ID_COLUMNS = ["shedid"]
WEIGHT_COLUMNS = ["weight", "weight_pop", "panel_weight", "panel_weight_pop"]

def mark_structural_skips(frame):
    result = frame.copy()
    protected = set(ID_COLUMNS + WEIGHT_COLUMNS)
    columns = [c for c in result.columns if c not in protected]
    for column in columns:
        if pd.api.types.is_numeric_dtype(result[column]):
            continue
        else:
            result[column] = result[column].astype("object").where(
                result[column].notna(), NOT_ASKED
            )
    return result

clean = mark_structural_skips(df)

print("Rows before:", len(df))
print("Rows after:", len(clean))

Rows before: 12934
Rows after: 12934


In [13]:
imputation_flags = [c for c in df.columns if c.endswith("_iflag")]

ef3_cols = ["EF3_a", "EF3_b", "EF3_c", "EF3_d", "EF3_e", "EF3_f", "EF3_g", "EF3_h"]

non_features = [
    "shedid", "duration",
    "weight", "weight_pop", "panel_weight", "panel_weight_pop","atleast_okay",
    "pay_casheqv","xlaptop", "field_month", "field_day", "control", "year",
]

excluded = imputation_flags + ef3_cols + non_features
print("Number of _iflag columns excluded:", len(imputation_flags))
print("Number of EF3 columns excluded:", len(ef3_cols))
print("Other non-feature columns excluded:", len(non_features))
print("Total columns excluded:", len(excluded))

new_features = clean.drop(columns=excluded, errors="ignore").copy()

print("Original:", df.shape)
print("Feature matrix:", new_features.shape)

Number of _iflag columns excluded: 357
Number of EF3 columns excluded: 8
Other non-feature columns excluded: 13
Total columns excluded: 378
Original: (12934, 815)
Feature matrix: (12934, 437)


features types 

In [14]:
categorical_features = new_features.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numerical_features = new_features.select_dtypes(include=["number"]).columns.tolist()

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))

Categorical: 414
Numerical: 23


In [15]:
for column in categorical_features:
    new_features[column] = (
        new_features[column]
        .str.strip()
        .replace("", np.nan)
    )

print("Categorical values cleaned.")
print("The __NOT_ASKED__ category was preserved.")

Categorical values cleaned.
The __NOT_ASKED__ category was preserved.


train and test 

In [16]:
new_features_train, new_features_test, target_train, target_test, weights_train, weights_test = train_test_split(
    new_features,
    fragility,
    population_weights,
    test_size=0.20,
    random_state=42,
    stratify=fragility
)

print("Train:", new_features_train.shape)
print("Test:", new_features_test.shape)

Train: (10347, 437)
Test: (2587, 437)


In [17]:
categorical_pipeline = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

numerical_pipeline = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", MinMaxScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_pipeline, categorical_features),
    ("num", numerical_pipeline, numerical_features)
])

X_train_prepared = preprocessor.fit_transform(new_features_train)
X_test_prepared = preprocessor.transform(new_features_test)

print("Train prepared:", X_train_prepared.shape)
print("Test prepared:", X_test_prepared.shape)

Train prepared: (10347, 1791)
Test prepared: (2587, 1791)


In [18]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_prepared, target_train)

predictions = model.predict(X_test_prepared)

print("Baseline accuracy:", accuracy_score(target_test, predictions))
print()
print(classification_report(target_test, predictions))

majority_baseline = (target_test == 0).mean()
print("Majority-class baseline (always predict 'not fragile'):", majority_baseline)


Baseline accuracy: 0.8569771936606108

              precision    recall  f1-score   support

           0       0.88      0.90      0.89      1680
           1       0.81      0.77      0.79       907

    accuracy                           0.86      2587
   macro avg       0.85      0.84      0.84      2587
weighted avg       0.86      0.86      0.86      2587

Majority-class baseline (always predict 'not fragile'): 0.6494008504058756
